# 04 — Tools

**Scope** — what the agent can *do*.

When you finish you can:

- give an agent your own Python functions to call with `@agent.tool_plain`
- write type hints and docstrings the model can read
- say "no" from inside a tool with `ModelRetry`, and set the retry budget
- prove which tools really ran by reading the messages

In [ ]:
from learningdantic import setup

MODEL = setup()

## A tool is a function the model may call

Tools set the agent's **scope**: it can only *do* what you hand it. The model never sees your code. It sees the
function's name, a schema built from your **type hints**, and your **docstring** as the description. The model
decides *when* to call a tool; your function decides *what happens*.

`@dice_agent.tool_plain` is a decorator (as in 03): it registers the function below it as a tool on that agent.
Your tools can be plain `def` functions. **Write the docstring for the model**, because it reads it.

In [ ]:
import random

from pydantic_ai import Agent

dice_agent = Agent(
    MODEL,
    instructions="You run a dice game. Use your tools; never invent rolls or player names.",
)


@dice_agent.tool_plain
def roll_dice() -> int:
    """Roll a six-sided die and return the result (1-6)."""
    return random.randint(1, 6)


@dice_agent.tool_plain
def get_player_name() -> str:
    """Get the name of the current player."""
    return "Anne"


result = await dice_agent.run(
    "Roll the dice, then tell me whether the player's guess of 4 beat it."
)
print(result.output)

> ✅ **Checkpoint** — an answer that names a roll (and usually Anne), for example `Anne’s guess of 4 beat the roll
> of 3.`
> The roll changes from run to run, because your `random.randint` really ran.
>
> 🔭 **In Logfire** (optional) — the trace shows the agent loop: a model call, then `running tool: roll_dice`, then
> a second model call that reads the tool's result.

Don't trust it? Every tool that ran leaves a `ToolReturnPart` in the run's messages. Collect their names:

This list comprehension reads as: for every message, for every part in it, keep `part.tool_name` if the part is
a `ToolReturnPart`.

In [ ]:
from pydantic_ai.messages import ToolReturnPart

tool_returns = [
    part.tool_name
    for message in result.all_messages()
    for part in message.parts
    if isinstance(part, ToolReturnPart)
]
print("tools that actually ran:", tool_returns)

> ✅ **Checkpoint** — you should see `'roll_dice'` (and usually `'get_player_name'`). That is proof the model
> called *your* functions and your Python actually ran.

## What the model was told about your tools

Earlier you read that the model sees only each tool's name, a schema from your type hints, and your
docstring. You can print exactly that, with no key and no cost:

- `TestModel(call_tools=[])` is a fake model (more in notebook 08) that calls none of your tools and just
  answers.
- `dice_agent.override(model=probe)` swaps that fake in for the indented lines of the `with` block, then
  puts your `MODEL` back. (`with x:` turns something on for the indented lines, then back off.)

After the run, `probe.last_model_request_parameters.function_tools` holds what the model was given.

In [ ]:
from pydantic_ai.models.test import TestModel

probe = TestModel(call_tools=[])
with dice_agent.override(model=probe):
    await dice_agent.run("Roll the dice.")  # answered by the fake, not your MODEL

for tool in probe.last_model_request_parameters.function_tools:
    print(f"{tool.name}: {tool.description}")
    print("   parameters:", tool.parameters_json_schema)

> ✅ **Checkpoint** — two tools, each with the docstring you wrote, for example
> `roll_dice: Roll a six-sided die and return the result (1-6).` Their `parameters` have empty
> `'properties'`, because these functions take no arguments. A type-hinted argument such as
> `country: str` would show up there as `'country': {'type': 'string'}`. That is all the model knows
> about your code.

## Saying no from inside a tool: `ModelRetry`

A tool is also where you enforce limits. If the model asks for something you can't handle, don't crash: raise
`ModelRetry("what to do instead")`, as in 03. The model reads your message and adjusts.

Same budget as 03: by default the model gets **1** retry, and if it keeps making a bad call the run raises
`UnexpectedModelBehavior`. `retries=3` on the agent gives every tool more room. (For one tool only, write
`@quiz_agent.tool_plain(retries=3)`.)

In [ ]:
from pydantic_ai import ModelRetry

capitals = {"france": "Paris", "japan": "Tokyo", "kenya": "Nairobi"}

quiz_agent = Agent(
    MODEL,
    instructions="You answer questions about capitals. Always use the lookup tool; never answer from memory.",
    retries=3,
)


@quiz_agent.tool_plain
def lookup_capital(country: str) -> str:
    """Look up the capital of one country. Only a few countries are in the database."""
    key = country.strip().lower()
    if key not in capitals:
        known = ", ".join(sorted(capitals))
        raise ModelRetry(
            f"'{country}' is not in the database. Known countries: {known}. "
            "Tell the user which countries you can answer for."
        )
    return capitals[key]


r = await quiz_agent.run("What are the capitals of France and Ruritania?")
print(r.output)

> ✅ **Checkpoint** — something like `France’s capital is Paris. Ruritania isn’t in the database; I can answer
> for France, Japan, and Kenya.` The model could only learn about Japan and Kenya from your `ModelRetry` message:
> it asked the tool for Ruritania, got your "no", and passed it on instead of making up a capital.

## 🏋️ Exercise — the temperature agent

1. Build `temp_agent` with two `@temp_agent.tool_plain` tools, `c_to_f(celsius: float) -> float` and
   `f_to_c(fahrenheit: float) -> float`, using the exact formulas. Ask it: "Is 25°C warmer than 80°F?"
2. Prove it used your tools: list the `ToolReturnPart`s in `result.all_messages()` (the pattern from above).

Stretch: if the agent did the maths itself, tighten its `instructions` until it always uses a tool.

Solution: [`../solutions/04_temperature_tools.ipynb`](../solutions/04_temperature_tools.ipynb)

## Next →

**[05 — Dependencies](05_dependencies.ipynb)**: hand your tools data at run time (and meet `RunContext`).